# Graphs II

In [1]:
import enum
class color(enum.Enum):
    White = 0
    Grey= 1
    Black = 2

## Dijkstra

Dijkstra is an algorithm to find the shortest path between two nodes of a **weighted** graph.

### Algorithm

```Python
G = {
1: [(2, 4), (3, 5)] # node 1 connected to node 2 with weight 4, to node 3 with weight 5
...
}

def dijkstra(G, s):
    dist = {s: 0}
    parent = {s: s}

    next_vertex = [s]
    while len(next_vertex):
        u = next_vertex.pop()
        for (v, w) in G[u]:
            if v not in dist or dist[v] > dist[u] + w:
                dist[v] = dist[u] + w
                parent[v] = u
                if v not in next_vertex:
                    next_vertex.append(v)
        next_vertex.sort(key=lambda x: -dist[x])
    return dist, parent
```

![](images/dijkstra.png)

### Shortest Distance Between all Nodes

If we want to find the shortest distance between all nodes, calling Dijktra for every starting node is too expensive.

For that use case, the Floyd-Warshall algorithm is a better choice, as it calculates the distance between all points using dynamic programming.

## Floyd-Warshall

This algorithm calculates the matrix $D^{(k)}[i,j]$ which is the smallest cost between nodes $i$ and $j$ using nodes numbered $k$ or less (assuming an arbitrary numbering of the nodes).

With this definition, we can see the following recursion:

$$ D^k [i,j] =
    \begin{cases}
        w_{i,j} & , k = 0 \\
        min(D^{k-1}[i,j],D^{k-1}[i,k],D^{k-1}[k,j]) & , \text{in other cases}
    \end{cases}
$$
where $w_{i,j}$ is the weight/cost between nodes $i$ and $j$.

### Implementation

In [2]:
from collections import defaultdict
from math import inf

def floyd_warshall(G):
    D0 = defaultdict(lambda: inf)
    D1 = defaultdict(lambda: inf)

    for (i, j, w) in G.edges():
        D0[i, j] = w

    for k in range(len(G.vertex)):
        for i, j in product(range(len(G.vertex)), repeat=2):
            D1[i, j] = min(D0[i, j], D0[i, k] + D0[k, j])
        D0, D1 = D1, D0
    return D0

In [11]:
from itertools import product


a = [0, 1, 2, 3]
print(list(product(a, repeat=2)))

[(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 1), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3), (3, 0), (3, 1), (3, 2), (3, 3)]


## Connected Components in Unidirected Graphs

An alternative to exploring DFS or BFS is to use the Union-Find Disjoint Set data structure. This datastructure is capable of very effiently merging elements marked as equal.

The main advantage is that the complexity of asking id something belongs to the same components is $O(1)$.

### Algorithm

In [2]:
class disjoint_set:
    def __init__(self):
        self.D = dict()

    def make_set(self, x):
        if x not in self.D:
            self.D[x] = [x, 0]
    
    def find(self, x):
        if x not in self.D:
            self.make_set(x)
        if self.D[x][0] != x:
            self.D[x] = self.find(self.D[x][0])
        return self.D[x]

    def is_same(self, x, y):
        return self.find(x) == self.find(y)
    
    def union(self, x, y):
        x_ = self.find(x)
        y_ = self.find(y)

        if x_[1] > y_[1]:
            y_[0] = x_[0]
        else:
            x_[0] = y_[0]
            if y_[1] == x_[1]:
                y_[1] += 1


### Example

![](images/cc/1.png)
![](images/cc/2.png)
![](images/cc/3.png)
![](images/cc/4.png)

## Minimum Spanning Tree (MST)

By running DFS or BFS, we can build a tree that touches all vertices of the graph. If the edges have weights, we are interested in finding the tree minimizes the total weights of the edges of the tree.

Two classical algorithms solve this type of problems:
- Prim's Algorithm
- Kruskal's Algorithm

### Prim's Algorithm

In [6]:
def prims(i, neighbors, nodes):
    next_vertex = [(i, 0)]
    u = nodes[i]
    u["color"] = color.Grey
    u["parent"] = None
    while len(next_vertex) > 0:
        i,_ = next_vertex.pop()
        v = nodes[i]
        v["color"] = color.Black
        for j, w in neighbors[i]:
            u = nodes[j]
            if u["color"] == color.White:
                u["color"] = color.Grey
                next_vertex.append((j, w))
                u["parent"] = i
            elif u["color"] == color.Grey:
                nv = next(nv for nv in next_vertex if nv[0] == j and nv[1] > w)
                nv[1] = w
                u["parent"] = i
        next_vertex.sort(key=lambda x: x[1], reverse=True)

#### Example

![](images/prims/1.png)
![](images/prims/2.png)
![](images/prims/3.png)
![](images/prims/4.png)
![](images/prims/5.png)

### Kruskal's Algorithm

Kruskal's algorithm iterates over edges instead of vertices, and uses disjoint-set-union datastructure to avoid creating loops.

In [7]:
def kruskal(neighbors):
    sorted_edges = sorted(
        ((u, v, w) for u in neighbors for (v, w) in neighbors[u]),
        key=lambda x: x[2],
        reverse=True
    )
    djs = disjoint_set()
    
    V = list()
    while len(sorted_edges) > 0:
        u, v, _ = sorted_edges.pop()
        if not djs.is_same(u, v):
            V.append((u, v))
            djs.union(u, v)
    return V

#### Example

![](images/kruskal/1.png)
![](images/kruskal/2.png)
![](images/kruskal/3.png)
![](images/kruskal/4.png)
![](images/kruskal/5.png)
![](images/kruskal/6.png)

## Max Flow Problems

Given a directed graph `G` with weighted edges that represent the flow capacity between nodes, find the maximum flow that can be sent from a source node `s`to a sink node `t`.

The steps to solve this problem are:
1. Run BFS from the `s` node until the `t` node is found, and record the path that was followed.
2. Send the maximum possible flow through that path and update the "remaining capacity" of the edges of the path.
3. Iterate until no new flow can be sent from `s` to `t`.

The algorithm that we just described is called the Edmon-Karp Algorithm.

In [8]:
def maxflow(G, s, t, edge_weights):
    def bfs(s, t):
        parent.clear()
        parent[s] = tuple() # something different than None
        next_node = [(s, 1E10)]

        # Find a path between s and t and calculate the
        # total flow that can be sent in that path
        while len(next_node) > 0:
            u, w = next_node.pop()

            for v in G[u]:
                if parent[v] is None and edge_weights[u, v]:
                    parent[v] = u;
                    new_flow = min(w, edge_weights[u, v])
                    if v == t:
                        return new_flow
                    next_node.append((v, new_flow))
        return 0

    flow = 0
    parent = defaultdict(lambda: None)

    while True:
        new_flow = bfs(s, t)
        if new_flow == 0:
            break
        flow += new_flow
        cur = t
        # update the remaining capacity of the path
        while cur != s:
            prev = parent[cur]
            edge_weights[prev, cur] -= new_flow
            edge_weights[cur, prev] += new_flow
            cur = prev
    return flow

### Example

![](images/flow/1.png)
![](images/flow/2.png)
![](images/flow/3.png)
![](images/flow/4.png)
![](images/flow/5.png)
![](images/flow/6.png)
![](images/flow/7.png)
![](images/flow/8.png)
![](images/flow/9.png)